# Math Assistant A2A Example

This notebook demonstrates how to use the NeMo Agent Toolkit SDK to create a math assistant that connects to a NAT-based calculator server via A2A protocol, showcasing NAT-to-NAT A2A communication with hybrid tool composition.

## Key Features

1. **A2A Protocol Integration** - Connect to a remote NAT calculator workflow via A2A
2. **MCP Client Integration** - Use MCP servers for time operations
3. **Custom Function Groups** - Use local logic evaluator tools
4. **Hybrid Tool Architecture** - Combine remote A2A, MCP, and local tools

## Prerequisites

1. Install the math assistant package:
   ```bash
   uv pip install -e examples/A2A/math_assistant_a2a
   ```

2. Start the NAT calculator A2A server:
   ```bash
   nat a2a serve --config_file examples/getting_started/simple_calculator/configs/config.yml --port 10000
   ```

3. Set environment variable:
   - `NVIDIA_API_KEY` - NVIDIA API key


In [ ]:
import os
import sys

# Add src to path for development
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)


## Creating the Workflow

We'll create a workflow that combines:
- An A2A client to connect to the NAT calculator server
- An MCP client to get current time information
- A logic evaluator for conditional operations


In [ ]:
from datetime import timedelta
from pathlib import Path

from pydantic import HttpUrl

from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.plugins.a2a.client.a2a_client import A2AClient
from nat.plugins.mcp.client_config import MCPClient
from nat.plugins.mcp.client_config import MCPServerConfig
from nat.plugins.mcp.client_config import MCPToolOverrideConfig
from nat.utils.sdk.nat_function_group import NatFunctionGroup
from nat.utils.sdk.nat_workflow import NatWorkflow

# Import the LogicEvaluator from the example package
from nat_math_assistant_a2a.register import LogicEvaluatorConfig


# Create SDK wrapper for LogicEvaluator
class LogicEvaluator(LogicEvaluatorConfig, NatFunctionGroup):
    """Logic evaluator function group for conditional operations."""
    pass


# Create the LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create A2A client to connect to the NAT calculator server
# Make sure the calculator server is running on port 10000
calculator_a2a = A2AClient(
    url=HttpUrl("http://localhost:10000"),
    task_timeout=timedelta(seconds=60),
    include_skills_in_description=True,
    name="calculator_a2a",
)

# Create MCP client for time operations
mcp_time = MCPClient(
    server=MCPServerConfig(
        transport="stdio",
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/Los_Angeles"],
    ),
    tool_overrides={
        "get_current_time": MCPToolOverrideConfig(
            alias="get_current_time_mcp",
            description="Get current date and time in Pacific timezone",
        ),
    },
    include=["get_current_time_mcp"],
    name="mcp_time",
)

# Create logic evaluator for conditional operations
logic_evaluator = LogicEvaluator(
    include=["if_then_else", "evaluate_condition"],
    name="logic_evaluator",
)

# Create the ReAct agent with all tools
agent = NatReActAgent(
    tools=[calculator_a2a, mcp_time, logic_evaluator],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

# Wrap in NatWorkflow
nat_workflow = NatWorkflow(
    entrypoint=agent,
)

print("Workflow created successfully!")


## Saving the Configuration

Save the workflow configuration to a YAML file.


In [ ]:
config_dir = Path(os.getcwd()) / "config"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "workflow_config.yaml"
nat_workflow.save_to_config_file(config_path)

print(f"Configuration saved to: {config_path}")
print("\n" + "="*50 + "\n")

with open(config_path) as f:
    print(f.read())


## Testing the Workflow

Test the workflow with a math query that combines calculation and time.

**Note**: The NAT calculator A2A server must be running for this to work:
```bash
nat a2a serve --config_file examples/getting_started/simple_calculator/configs/config.yml --port 10000
```


In [ ]:
# Test the workflow (requires NAT calculator A2A server to be running)
# Uncomment to run:
# result = await nat_workflow.prompt(
#     "Is the product of 2 and 4 greater than the current hour of the day?"
# )
# print(result)


In [ ]:
# Additional test queries
# Uncomment to run:

# Basic calculation
# result = await nat_workflow.prompt("What is 25 multiplied by 4?")
# print(result)

# Time-integrated calculation
# result = await nat_workflow.prompt("Add the current hour to 100")
# print(result)

# Multi-step problem
# result = await nat_workflow.prompt(
#     "If 15 divided by 3 is greater than 4, what is 10 times 5?"
# )
# print(result)


## Summary

This notebook demonstrated:

1. **A2AClient SDK Class** - Connecting to a NAT-based calculator server via A2A protocol
2. **MCPClient SDK Class** - Using MCP servers for time operations
3. **Custom Function Groups** - Creating SDK wrappers for custom function groups
4. **Hybrid Architecture** - Combining remote A2A, MCP, and local tools

### Tool Composition

The workflow combines three types of tools:

| Tool Type | Name | Description |
|---|---|---|
| A2A Client | `calculator_a2a` | Remote NAT calculator (add, subtract, multiply, divide, compare) |
| MCP Client | `mcp_time` | Local MCP server for time operations |
| Function Group | `logic_evaluator` | Local logic evaluator (if_then_else, evaluate_condition) |

### Related Examples

- [Currency Agent A2A](../currency_agent_a2a/) - External A2A service integration
- [Simple Calculator](../../getting_started/simple_calculator/) - The calculator workflow used as A2A server
